# 02_Fingerprint Preprocessing

## Objective
The objective of this notebook is to preprocess and enhance fingerprint images for robust feature extraction.

## Method
Fingerprint preprocessing/enhancement pipeline.

## Input
Raw fingerprint images from the inspection stage.

## Processing Tasks

1. Load and Validate Fingerprint Images.
2. CLAHE Image Enhancement.
3. Original vs Enhanced Fingerprint Comparison.
4. Enhanced Fingerprint Images Validation.

## Output
- Processed fingerprint images.
- Visual enhancement comparisons.
- Prepared inputs for feature extraction.

## Purpose
Improve fingerprint image clarity and consistency for feature extraction.


In [ ]:

from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_PATH = Path(r"E:\Multimodal Biometric Verification System")

INPUT_PATH = PROJECT_PATH / "Dataset" / "Raw" / "Fingerprint"

OUTPUT_PATH = (
    PROJECT_PATH / "Dataset" / "Processed" / "Fingerprint" / "Fingerprint_CLAHE"
)
USERS = ["User_01", "User_02"]

for user_id in USERS:
    (OUTPUT_PATH / user_id).mkdir(parents=True, exist_ok=True)

print("Fingerprint Preprocessing Environment Configured")
print("---------------------------------------------")
print(f"Input Path: {INPUT_PATH}")
print(f"Output Path: {OUTPUT_PATH}")
print(f"Users: {USERS}")


### 1. Load and Validate Fingerprint Images

In this section, we load the original fingerprint images from both users and validate their readability, dimensions, and data types.

The original images are loaded in grayscale format without modifying or overwriting the source files.


In [ ]:

fingerprint_images = {}
validation_records = []

for user_id in USERS:
    user_path = INPUT_PATH / user_id

    image_paths = sorted(user_path.glob("*.BMP"))

    for image_path in image_paths:
        image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)

        if image is None:
            print(f"Failed to load: {image_path.name}")
            continue

        fingerprint_images[(user_id, image_path.name)] = image

        validation_records.append({
            "User_ID": user_id,
            "Image_Name": image_path.name,
            "Height": image.shape[0],
            "Width": image.shape[1],
            "Data_Type": str(image.dtype),
            "Min_Pixel_Value": int(image.min()),
            "Max_Pixel_Value": int(image.max())
        })

validation_df = pd.DataFrame(validation_records)

print("Fingerprint Images Loaded Successfully")
print("---------------------------------------------")
print(f"Total Images Loaded: {len(fingerprint_images)}")
print(f"User_01 Images: {(validation_df['User_ID'] == 'User_01').sum()}")
print(f"User_02 Images: {(validation_df['User_ID'] == 'User_02').sum()}")
print(f"Unreadable Images: {20 - len(fingerprint_images)}")

print("\nImage Dimensions:")
print(validation_df.groupby("User_ID")[["Height", "Width"]].value_counts())

print("\nData Types:")
print(validation_df["Data_Type"].value_counts())

print("\nPixel Value Range:")
print(f"Minimum: {validation_df['Min_Pixel_Value'].min()}")
print(f"Maximum: {validation_df['Max_Pixel_Value'].max()}")

print("\nValidation Records:")
display(validation_df)


### 2. CLAHE Image Enhancement


In [ ]:

CLAHE_CLIP_LIMIT = 2.0
CLAHE_TILE_GRID_SIZE = (8, 8)

clahe = cv2.createCLAHE(
    clipLimit=CLAHE_CLIP_LIMIT,
    tileGridSize=CLAHE_TILE_GRID_SIZE
)

enhanced_images = {}
enhancement_records = []

for (user_id, image_name), original_image in fingerprint_images.items():

    enhanced_image = clahe.apply(original_image)

    enhanced_images[(user_id, image_name)] = enhanced_image

    output_file = OUTPUT_PATH / user_id / image_name
    cv2.imwrite(str(output_file), enhanced_image)

    enhancement_records.append({
        "User_ID": user_id,
        "Image_Name": image_name,
        "Original_Mean": round(float(np.mean(original_image)), 2),
        "Enhanced_Mean": round(float(np.mean(enhanced_image)), 2),
        "Original_Std": round(float(np.std(original_image)), 2),
        "Enhanced_Std": round(float(np.std(enhanced_image)), 2)
    })

enhancement_df = pd.DataFrame(enhancement_records)

print("CLAHE Enhancement Completed")
print("---------------------------------------------")
print(f"Images Enhanced: {len(enhanced_images)}")
print(f"User_01: {(enhancement_df['User_ID'] == 'User_01').sum()}")
print(f"User_02: {(enhancement_df['User_ID'] == 'User_02').sum()}")
print(f"Output Directory: {OUTPUT_PATH}")

print("\nEnhancement Statistics:")
display(enhancement_df)

print("\nAverage Statistics Before and After CLAHE:")
display(
    enhancement_df[
        ["Original_Mean", "Enhanced_Mean", "Original_Std", "Enhanced_Std"]
    ].mean().to_frame("Average_Value")
)


### 3. Original vs Enhanced Fingerprint Comparison


In [ ]:

finger_order = [
    "Left_index_finger",
    "Left_little_finger",
    "Left_middle_finger",
    "Left_ring_finger",
    "Left_thumb_finger",
    "Right_index_finger",
    "Right_little_finger",
    "Right_middle_finger",
    "Right_ring_finger",
    "Right_thumb_finger"
]

fig, axes = plt.subplots(2, 10, figsize=(22, 7))

for row, user_id in enumerate(USERS):
    for col, finger in enumerate(finger_order):

        image_name = f"{user_id}_{finger}.BMP"
        key = (user_id, image_name)

        original_image = fingerprint_images[key]
        enhanced_image = enhanced_images[key]

        axes[row, col].imshow(enhanced_image, cmap="gray")
        axes[row, col].set_title(
            finger.replace("_", "\n"),
            fontsize=8
        )
        axes[row, col].axis("off")

    axes[row, 0].set_ylabel(
        f"{user_id}\nEnhanced",
        fontsize=11,
        labelpad=12
    )

plt.suptitle("CLAHE Enhanced Fingerprint Images", fontsize=15)
plt.tight_layout()
plt.show()

# Original images
fig, axes = plt.subplots(2, 10, figsize=(22, 7))

for row, user_id in enumerate(USERS):
    for col, finger in enumerate(finger_order):

        image_name = f"{user_id}_{finger}.BMP"
        original_image = fingerprint_images[(user_id, image_name)]

        axes[row, col].imshow(original_image, cmap="gray")
        axes[row, col].set_title(
            finger.replace("_", "\n"),
            fontsize=8
        )
        axes[row, col].axis("off")

    axes[row, 0].set_ylabel(
        f"{user_id}\nOriginal",
        fontsize=11,
        labelpad=12
    )

plt.suptitle("Original Fingerprint Images", fontsize=15)
plt.tight_layout()
plt.show()

print("Visual Comparison Completed")
print("---------------------------------------------")
print(f"Original Images: {len(fingerprint_images)}")
print(f"Enhanced Images: {len(enhanced_images)}")
print("Image Dimensions: 103 x 96")


### 4. Enhanced Fingerprint Images Validation


In [ ]:
validation_records = []
unreadable_images = []

for user_id in USERS:
    user_path = OUTPUT_PATH / user_id

    image_files = sorted(user_path.glob("*.BMP"))

    for image_file in image_files:
        image = cv2.imread(str(image_file), cv2.IMREAD_GRAYSCALE)

        if image is None:
            unreadable_images.append(str(image_file))
            continue

        validation_records.append({
            "User_ID": user_id,
            "Image_Name": image_file.name,
            "Height": image.shape[0],
            "Width": image.shape[1],
            "Data_Type": str(image.dtype),
            "Min_Pixel_Value": int(image.min()),
            "Max_Pixel_Value": int(image.max())
        })

validation_df = pd.DataFrame(validation_records)

print("Enhanced Fingerprint Validation Completed")
print("---------------------------------------------")
print(f"Total Enhanced Images: {len(validation_df)}")

print("\nImages per User:")
print(validation_df["User_ID"].value_counts().sort_index())

print(f"\nUnreadable Images: {len(unreadable_images)}")

print("\nImage Dimensions:")
print(validation_df[["Height", "Width"]].value_counts())

print("\nData Types:")
print(validation_df["Data_Type"].value_counts())

print("\nMissing or Invalid Images:")
print(unreadable_images if unreadable_images else "None")

print("\nValidation Preview:")
display(validation_df.head())